# 07_Trigger_Cloud_Run_Embedding_Job

Tareas:

1. Dispara el Cloud Run Job encargado de generar embeddings para los chunks pendientes almacenados en Supabase.
2. Espera a que la ejecución termine y valida que los chunks del corpus actual hayan quedado embebidos correctamente.

In [0]:
# Dependencias
%pip install -q google-auth requests pg8000


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# ============================================================
# Configuración
# ============================================================

from __future__ import annotations

from datetime import datetime, timezone
import json
import time
import uuid

import pg8000.dbapi
import requests
from google.oauth2 import service_account
from google.auth.transport.requests import Request
from pyspark.sql import functions as F
from pyspark.sql import types as T

# Databricks
CATALOG_NAME = "workspace"
SCHEMA_NAME = "tfm_pmc"
INVENTORY_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.pmc_inventory"
PIPELINE_RUNS_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.pipeline_runs"

PIPELINE_NAME = "07_Trigger_Cloud_Run_Embedding_Job_v2_Clean"
RUN_ID = str(uuid.uuid4())
RUN_STARTED_AT = datetime.now(timezone.utc)

# Google Cloud Run
GCP_PROJECT_ID = "tfm-complutense-506805"
GCP_REGION = "us-central1"
CLOUD_RUN_JOB_NAME = "tfm-embeddings"

DATABRICKS_SECRET_SCOPE = "tfm"
GCP_SERVICE_ACCOUNT_SECRET_KEY = "gcp-service-account"

# Supabase / PostgreSQL
SUPABASE_HOST = "aws-0-ca-central-1.pooler.supabase.com"
SUPABASE_PORT = 6543
SUPABASE_DATABASE = "postgres"
SUPABASE_USER = "postgres.cwymrqsnzgoyzjmhdrkz"
SUPABASE_PASSWORD_SECRET_KEY = "supabase-password"

# Polling
POLL_INTERVAL_SECONDS = 10
MAX_WAIT_SECONDS = 30 * 60

print("Run ID:", RUN_ID)


Run ID: be2712f5-2c8a-4c6f-b853-7998c46a3a4a


In [0]:
# ============================================================
# Helpers de autenticación y conexión
# ============================================================


def get_google_credentials():
    service_account_json = dbutils.secrets.get(
        scope=DATABRICKS_SECRET_SCOPE,
        key=GCP_SERVICE_ACCOUNT_SECRET_KEY,
    )

    service_account_info = json.loads(service_account_json)

    credentials = service_account.Credentials.from_service_account_info(
        service_account_info,
        scopes=["https://www.googleapis.com/auth/cloud-platform"],
    )

    credentials.refresh(Request())
    return credentials


def get_supabase_connection():
    password = dbutils.secrets.get(
        scope=DATABRICKS_SECRET_SCOPE,
        key=SUPABASE_PASSWORD_SECRET_KEY,
    )

    return pg8000.dbapi.connect(
        host=SUPABASE_HOST,
        port=SUPABASE_PORT,
        database=SUPABASE_DATABASE,
        user=SUPABASE_USER,
        password=password,
    )


def get_auth_headers(credentials):
    if not credentials.valid or credentials.expired:
        credentials.refresh(Request())

    return {
        "Authorization": f"Bearer {credentials.token}",
        "Content-Type": "application/json",
    }


In [0]:
# ============================================================
# Determinar corpus actual y estado de embeddings
# ============================================================

current_documents = (
    spark.table(INVENTORY_TABLE)
    .filter(
        (F.col("is_latest_version") == True)
        & (F.col("publication_status") == "completed")
    )
    .select("pmcid")
    .distinct()
    .orderBy("pmcid")
)

current_pmcids = [
    row["pmcid"]
    for row in current_documents.collect()
]

if not current_pmcids:
    raise RuntimeError(
        "No hay documentos publicados en el corpus actual. "
        "Ejecuta primero el Script 06."
    )

print("Documents in current corpus:", len(current_pmcids))


def get_embedding_status(pmcids):
    placeholders = ", ".join(["%s"] * len(pmcids))

    query = f"""
        SELECT
            COUNT(*) AS total_chunks,
            COUNT(*) FILTER (
                WHERE c.embedding_status = 'completed'
                  AND c.embedding IS NOT NULL
            ) AS completed_chunks,
            COUNT(*) FILTER (
                WHERE c.embedding_status = 'pending'
            ) AS pending_chunks,
            COUNT(*) FILTER (
                WHERE c.embedding_status = 'failed'
            ) AS failed_chunks,
            COUNT(*) FILTER (
                WHERE c.embedding IS NULL
            ) AS missing_embeddings
        FROM rag.document_chunks c
        JOIN rag.documents d
          ON d.document_id = c.document_id
        WHERE d.pmc_id IN ({placeholders})
    """

    connection = get_supabase_connection()
    cursor = connection.cursor()

    try:
        cursor.execute(query, tuple(pmcids))
        row = cursor.fetchone()
    finally:
        cursor.close()
        connection.close()

    return {
        "total_chunks": int(row[0] or 0),
        "completed_chunks": int(row[1] or 0),
        "pending_chunks": int(row[2] or 0),
        "failed_chunks": int(row[3] or 0),
        "missing_embeddings": int(row[4] or 0),
    }


status_before = get_embedding_status(current_pmcids)
print("Embedding status before Cloud Run:")
print(json.dumps(status_before, indent=2))

if status_before["total_chunks"] == 0:
    raise RuntimeError(
        "No se encontraron chunks en Supabase para el corpus actual."
    )


Documents in current corpus: 20
Embedding status before Cloud Run:
{
  "total_chunks": 643,
  "completed_chunks": 0,
  "pending_chunks": 0,
  "failed_chunks": 643,
  "missing_embeddings": 643
}


In [0]:
# ============================================================
# Disparar Cloud Run Job y esperar finalización
# ============================================================


def trigger_cloud_run_job(credentials):
    url = (
        "https://run.googleapis.com/v2/"
        f"projects/{GCP_PROJECT_ID}/"
        f"locations/{GCP_REGION}/"
        f"jobs/{CLOUD_RUN_JOB_NAME}:run"
    )

    response = requests.post(
        url,
        headers=get_auth_headers(credentials),
        json={},
        timeout=30,
    )

    response.raise_for_status()
    operation = response.json()

    operation_name = operation.get("name")
    if not operation_name:
        raise RuntimeError(
            "Cloud Run aceptó la solicitud, pero no devolvió operation.name."
        )

    print("Cloud Run job triggered successfully.")
    print("Operation:", operation_name)

    return operation_name


def wait_for_operation(credentials, operation_name):
    operation_url = f"https://run.googleapis.com/v2/{operation_name}"
    started = time.time()

    while True:
        response = requests.get(
            operation_url,
            headers=get_auth_headers(credentials),
            timeout=30,
        )
        response.raise_for_status()

        operation = response.json()

        if operation.get("done"):
            if operation.get("error"):
                raise RuntimeError(
                    "Cloud Run embedding job failed: "
                    + json.dumps(operation["error"], ensure_ascii=False)
                )

            print("Cloud Run execution finished successfully.")
            return operation

        elapsed = int(time.time() - started)

        if elapsed >= MAX_WAIT_SECONDS:
            raise TimeoutError(
                f"Cloud Run did not finish within {MAX_WAIT_SECONDS} seconds."
            )

        print(
            f"Cloud Run still running... "
            f"elapsed={elapsed}s"
        )
        time.sleep(POLL_INTERVAL_SECONDS)


credentials = get_google_credentials()
operation_name = None
operation_result = None

if (
    status_before["pending_chunks"] == 0
    and status_before["missing_embeddings"] == 0
    and status_before["failed_chunks"] == 0
):
    print("No pending embeddings. Cloud Run will not be triggered.")
else:
    operation_name = trigger_cloud_run_job(credentials)
    operation_result = wait_for_operation(
        credentials,
        operation_name,
    )


Cloud Run job triggered successfully.
Operation: projects/tfm-complutense-506805/locations/us-central1/operations/b2126bfc-97ba-4d96-bb3e-49719bc647f0
Cloud Run still running... elapsed=0s
Cloud Run still running... elapsed=10s
Cloud Run still running... elapsed=20s
Cloud Run still running... elapsed=31s
Cloud Run still running... elapsed=41s
Cloud Run still running... elapsed=52s
Cloud Run still running... elapsed=62s
Cloud Run still running... elapsed=72s
Cloud Run still running... elapsed=82s
Cloud Run still running... elapsed=93s
Cloud Run still running... elapsed=103s
Cloud Run still running... elapsed=114s
Cloud Run still running... elapsed=124s
Cloud Run still running... elapsed=134s
Cloud Run still running... elapsed=145s
Cloud Run still running... elapsed=155s
Cloud Run still running... elapsed=165s
Cloud Run still running... elapsed=175s
Cloud Run still running... elapsed=186s
Cloud Run still running... elapsed=196s
Cloud Run still running... elapsed=206s
Cloud Run still runn

In [0]:
# ============================================================
# Validación final de embeddings en Supabase
# ============================================================

status_after = get_embedding_status(current_pmcids)

print("===== EMBEDDING VALIDATION =====")
print("Total chunks:       ", status_after["total_chunks"])
print("Completed:          ", status_after["completed_chunks"])
print("Pending:            ", status_after["pending_chunks"])
print("Failed:             ", status_after["failed_chunks"])
print("Missing embeddings: ", status_after["missing_embeddings"])

embedding_validation_ok = (
    status_after["total_chunks"] > 0
    and status_after["pending_chunks"] == 0
    and status_after["failed_chunks"] == 0
    and status_after["missing_embeddings"] == 0
    and status_after["completed_chunks"] == status_after["total_chunks"]
)

if not embedding_validation_ok:
    raise RuntimeError(
        "La ejecución de Cloud Run terminó, pero la validación de embeddings "
        "del corpus actual no fue satisfactoria. Revisa los conteos anteriores."
    )

print("Embedding pipeline completed successfully.")


===== EMBEDDING VALIDATION =====
Total chunks:        643
Completed:           643
Pending:             0
Failed:              0
Missing embeddings:  0
Embedding pipeline completed successfully.


In [0]:
# ============================================================
# Registrar ejecución en Databricks
# ============================================================

RUN_COMPLETED_AT = datetime.now(timezone.utc)

run_metadata = {
    "gcp_project_id": GCP_PROJECT_ID,
    "gcp_region": GCP_REGION,
    "cloud_run_job_name": CLOUD_RUN_JOB_NAME,
    "cloud_run_operation": operation_name,
    "cloud_run_triggered": operation_name is not None,
    "poll_interval_seconds": POLL_INTERVAL_SECONDS,
    "max_wait_seconds": MAX_WAIT_SECONDS,
    "documents_validated": len(current_pmcids),
    "status_before": status_before,
    "status_after": status_after,
}

run_row = spark.createDataFrame(
    [(
        RUN_ID,
        PIPELINE_NAME,
        "completed",
        RUN_STARTED_AT,
        RUN_COMPLETED_AT,
        int(status_after["total_chunks"]),
        int(status_after["total_chunks"]),
        int(status_after["completed_chunks"]),
        0,
        0,
        0,
        json.dumps(run_metadata),
        None,
    )],
    schema=T.StructType([
        T.StructField("run_id", T.StringType(), False),
        T.StructField("pipeline_name", T.StringType(), False),
        T.StructField("run_status", T.StringType(), False),
        T.StructField("started_at", T.TimestampType(), False),
        T.StructField("completed_at", T.TimestampType(), True),
        T.StructField("records_requested", T.LongType(), True),
        T.StructField("records_found", T.LongType(), True),
        T.StructField("records_processed", T.LongType(), True),
        T.StructField("records_inserted", T.LongType(), True),
        T.StructField("records_updated", T.LongType(), True),
        T.StructField("records_failed", T.LongType(), True),
        T.StructField("execution_metadata", T.StringType(), True),
        T.StructField("error_message", T.StringType(), True),
    ]),
)

run_row.write.mode("append").saveAsTable(PIPELINE_RUNS_TABLE)

print("Pipeline run registered in Databricks.")


Pipeline run registered in Databricks.


In [0]:
# ============================================================
# Auditoría final
# ============================================================

print("===== CURRENT CORPUS =====")
display(current_documents)

print("===== FINAL EMBEDDING STATUS =====")
status_after_df = spark.createDataFrame(
    [(
        status_after["total_chunks"],
        status_after["completed_chunks"],
        status_after["pending_chunks"],
        status_after["failed_chunks"],
        status_after["missing_embeddings"],
    )],
    [
        "total_chunks",
        "completed_chunks",
        "pending_chunks",
        "failed_chunks",
        "missing_embeddings",
    ],
)
display(status_after_df)

print("===== DATABRICKS PIPELINE RUN =====")
display(
    spark.table(PIPELINE_RUNS_TABLE)
    .filter(F.col("run_id") == RUN_ID)
)


===== CURRENT CORPUS =====


pmcid
PMC13538227
PMC13538279
PMC13539412
PMC13540131
PMC13543812
PMC13544149
PMC13545448
PMC13547081
PMC13547469
PMC13552502


===== FINAL EMBEDDING STATUS =====


total_chunks,completed_chunks,pending_chunks,failed_chunks,missing_embeddings
643,643,0,0,0


===== DATABRICKS PIPELINE RUN =====


run_id,pipeline_name,run_status,started_at,completed_at,records_requested,records_found,records_processed,records_inserted,records_updated,records_failed,execution_metadata,error_message
be2712f5-2c8a-4c6f-b853-7998c46a3a4a,07_Trigger_Cloud_Run_Embedding_Job_v2_Clean,completed,2026-09-14T00:13:26.558Z,2026-09-14T00:17:51.910Z,643,643,643,0,0,0,"{""gcp_project_id"": ""tfm-complutense-506805"", ""gcp_region"": ""us-central1"", ""cloud_run_job_name"": ""tfm-embeddings"", ""cloud_run_operation"": ""projects/tfm-complutense-506805/locations/us-central1/operations/b2126bfc-97ba-4d96-bb3e-49719bc647f0"", ""cloud_run_triggered"": true, ""poll_interval_seconds"": 10, ""max_wait_seconds"": 1800, ""documents_validated"": 20, ""status_before"": {""total_chunks"": 643, ""completed_chunks"": 0, ""pending_chunks"": 0, ""failed_chunks"": 643, ""missing_embeddings"": 643}, ""status_after"": {""total_chunks"": 643, ""completed_chunks"": 643, ""pending_chunks"": 0, ""failed_chunks"": 0, ""missing_embeddings"": 0}}",null
